# 01 Data Prep

End-to-end data pipeline (matches `docs/claude-3.md` Step 6):

1. Optional: install Kaggle CLI locally (`pip install kaggle`) and download raw novels.
2. On Kaggle: clone this repo, attach the **Jinyong Wuxia** dataset as input, then run the code cells.
3. `clean_text.py` — raw `data/raw/*.txt` → cleaned `data/processed/*.txt`.
4. `build_instructions.py` — cleaned text → `data/instructions/jinyong_sft.jsonl`.
5. Inspect samples; upload JSONL as a Kaggle Dataset for `02_train.ipynb`.

## Resolve repository root
Works when the notebook lives in `notebooks/` or the repo root.

In [2]:
from pathlib import Path
import os


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "clean_text.py").is_file():
            return cand
    raise FileNotFoundError("Could not find scripts/clean_text.py; cd into jinyong-finetune")


REPO = find_repo_root()
os.chdir(REPO)
print("REPO:", REPO)

REPO: /Users/william.jiang/my-tests/jinyong-finetune


## Kaggle-only (uncomment)
```
# !pip install -q pyyaml
# !git clone https://github.com/jxjwilliam/jinyong-finetune.git /kaggle/working/jinyong-finetune
# %cd /kaggle/working/jinyong-finetune
```
Mount the Kaggle dataset so `data/raw/` contains the `.txt` novels (or set `--input-dir`).

In [3]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/clean_text.py", "--dry-run"], check=False)

[DRY RUN] cn_stopwords.txt               utf-8         2,079 →      2,078 chars (-0.0%)
[DRY RUN] jinyong.txt                    utf-8     8,699,641 →  5,895,221 chars (-32.2%)
[DRY RUN] person.txt                     utf-8         6,879 →      5,457 chars (-20.7%)
[DRY RUN] person_count.txt               utf-8        13,533 →      9,410 chars (-30.5%)

Total: 8,722,132 → 5,912,166 chars


CompletedProcess(args=['/Users/william.jiang/my-tests/jinyong-finetune/venv/bin/python', 'scripts/clean_text.py', '--dry-run'], returncode=0)

In [4]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/clean_text.py"], check=False)

cn_stopwords.txt               utf-8         2,079 →      2,078 chars (-0.0%)
jinyong.txt                    utf-8     8,699,641 →  5,895,221 chars (-32.2%)
person.txt                     utf-8         6,879 →      5,457 chars (-20.7%)
person_count.txt               utf-8        13,533 →      9,410 chars (-30.5%)

Total: 8,722,132 → 5,912,166 chars


CompletedProcess(args=['/Users/william.jiang/my-tests/jinyong-finetune/venv/bin/python', 'scripts/clean_text.py'], returncode=0)

In [5]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/build_instructions.py", "--stats"], check=False)

cn_stopwords.txt: 8 sliding windows
jinyong.txt: 29474 sliding windows
person.txt: 25 sliding windows
person_count.txt: 45 sliding windows
Continuation pairs:  29,552
Typed scene pairs:   3,694
Total pairs:         33,246
Train / Val split:   31,584 / 1,662  (eval_ratio=0.05, seed=42)
Saved → data/instructions/jinyong_sft.jsonl


CompletedProcess(args=['/Users/william.jiang/my-tests/jinyong-finetune/venv/bin/python', 'scripts/build_instructions.py', '--stats'], returncode=0)

## Inspect JSONL samples

In [6]:
import json
import random
from pathlib import Path

p = Path("data/instructions/jinyong_sft.jsonl")
if not p.is_file():
    print("Run build_instructions first.")
else:
    lines = p.read_text(encoding="utf-8").splitlines()
    sample = random.sample(lines, k=min(5, len(lines)))
    for row in sample:
        obj = json.loads(row)
        print("instruction:", obj["instruction"][:60], "…")
        print("input len:", len(obj["input"]), "output len:", len(obj["output"]))
        print("---")

instruction: 以金庸武侠小说的风格，续写以下段落： …
input len: 300 output len: 300
---
instruction: 以金庸武侠小说的风格，续写以下段落： …
input len: 300 output len: 300
---
instruction: 以金庸武侠小说的风格，续写以下段落： …
input len: 300 output len: 300
---
instruction: 以金庸武侠小说的风格，续写以下段落： …
input len: 300 output len: 300
---
instruction: 以金庸武侠小说的风格，续写以下段落： …
input len: 300 output len: 300
---
